# Competitiveness model diagnostics

Sanity checks for the multi-quantile model on the 2024 T-60 full feature set:

1. **Calibration curve** — predicted P(close) bucketed vs actual close-race rate per bucket.
2. **Q-Q plot of margin residuals** — does the predicted q=0.5 (median margin) look sensible vs actual?
3. **Per-rating-bucket residual std** — Tossup races should have wider residuals than Solid X.
4. **Quantile crossing rate** — fraction of test rows where adjacent quantiles came back in wrong order.

If any of these look weird, the headline numbers in `03_backtest_curves.ipynb` are suspect.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, '../src')
from oath_score.scores.competitiveness import LogisticCompetitiveness, SCORE_COL
from oath_score.scores.multi_quantile import MultiQuantileCompetitiveness

PROC = Path('../data/processed')
train = pd.concat([pd.read_parquet(PROC / f'candidates_{c}_T-60.parquet') for c in (2014, 2016, 2022)],
                  ignore_index=True)
test = pd.read_parquet(PROC / 'candidates_2024_T-60.parquet')
print(f'train: {len(train)} rows, test: {len(test)} rows')

## 1. Calibration curve

In [ ]:
model = MultiQuantileCompetitiveness(feature_set_name='full').fit(train)
scored = model.score(test)
dems = scored[scored['party_major'] == 'D'].copy()
dems['actual_close'] = (dems['margin_pct'].abs() < 0.05).astype(int)

# Bucket predicted P into quantile bins; compute actual rate per bucket
bins = np.linspace(0, dems[SCORE_COL].max() + 0.01, 8)
dems['pred_bucket'] = pd.cut(dems[SCORE_COL], bins=bins)
calib = dems.groupby('pred_bucket', observed=True).agg(
    n=('actual_close', 'size'),
    mean_pred=(SCORE_COL, 'mean'),
    rate_actual=('actual_close', 'mean'),
)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(calib['mean_pred'], calib['rate_actual'], s=calib['n']*4, alpha=0.7)
for _, r in calib.iterrows():
    ax.annotate(f"n={int(r['n'])}", (r['mean_pred'], r['rate_actual']),
                fontsize=8, ha='center', va='bottom')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='perfect calibration')
ax.set_xlabel('Mean predicted P(|margin|<0.05)')
ax.set_ylabel('Actual close-race rate in bucket')
ax.set_title('Calibration: multi-quantile + full @ 2024 T-60')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.0)
ax.set_ylim(-0.02, 1.0)
plt.tight_layout()
calib

## 2. Q-Q plot of predicted q=0.5 vs actual margin

If the median quantile is well-calibrated, predicted-vs-actual should hug the diagonal across the margin range.

In [ ]:
# Use a slim helper to recover the median margin prediction directly
X = model._featurize(dems)
X_scaled = model._scaler.transform(X)
q50_idx = list(model.quantiles).index(0.5)
predicted_margin = model._models[q50_idx].predict(X_scaled)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(predicted_margin, dems['margin_pct'], alpha=0.4, s=20)
lim = max(abs(predicted_margin).max(), abs(dems['margin_pct']).max())
ax.plot([-lim, lim], [-lim, lim], 'k--', alpha=0.5)
ax.axhline(0, color='gray', alpha=0.3)
ax.axvline(0, color='gray', alpha=0.3)
ax.set_xlabel('Predicted median margin')
ax.set_ylabel('Actual signed margin (D% - R%)')
ax.set_title('Q-Q-style: predicted q=0.5 vs actual margin (2024 Dems)')
ax.grid(True, alpha=0.3)
plt.tight_layout()

## 3. Residual std by Cook rating bucket

Tossup races should have wider residuals than Solid X — those races are by definition harder to predict. If the residuals are flat across buckets, the model isn't using cook_rating's uncertainty signal.

In [ ]:
dems['residual'] = dems['margin_pct'] - predicted_margin
by_bucket = dems[dems['cook_rating'].notna()].groupby('cook_rating')['residual'].agg(
    n='size', std='std', mean='mean'
).round(3)
by_bucket

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(by_bucket.index, by_bucket['std'], color='#3366cc')
ax.set_xlabel('Cook rating (1=Solid R, 7=Solid D)')
ax.set_ylabel('Residual std')
ax.set_title('Residual variance by Cook rating bucket')
ax.grid(True, alpha=0.3)
plt.tight_layout()

## 4. Quantile crossing rate

In [ ]:
print(f'Multi-quantile crossing rate (2024 T-60 full): {model.crossing_rate:.3f}')
print(f'  → {int(model.crossing_rate * len(dems))} of {len(dems)} test rows had at least one crossing.')
print('Crossings are repaired in-place by per-row sort before computing P(|margin|<0.05).')

## 5. Logistic-model coefficients (sanity check signs)

Standardized coefficients on the full feature set. Expected: positive on `incumbent` would mean incumbents are MORE likely to be in close races (counterintuitive — they tend to win bigger). Negative would mean less likely (sensible). Positive on `self_raised_log` and `opp_raised_log` — fundraising correlates with competitiveness. Negative on `cpvi` magnitude, etc.

In [ ]:
logistic_full = LogisticCompetitiveness(feature_set_name='full').fit(train)
coefs = pd.Series(logistic_full.coef_).sort_values(key=abs, ascending=False)
coefs.to_frame('standardized_coef').style.format({'standardized_coef': '{:+.3f}'})